# 🏥 MediScan AI — Medical Image & Report Assistant

**Stack:** OpenCV · PyTorch · Hugging Face · Scikit-learn · NLTK · LLaMA (NVIDIA API) · FastAPI

**Pipeline:**
1. Upload a medical scan (X-ray, CT, MRI)
2. OpenCV preprocesses the image
3. Hugging Face + PyTorch classify and extract features
4. Scikit-learn scores anomalies
5. NLTK formats findings into a structured prompt
6. LLaMA (via NVIDIA API) generates a natural-language radiology report
7. FastAPI exposes the full pipeline as a REST endpoint

## Cell 1 — Install dependencies
Run once. Comment out after first run.

In [1]:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# !pip install transformers scikit-learn opencv-python-headless nltk
# !pip install fastapi uvicorn nest-asyncio python-multipart Pillow requests python-dotenv

## Cell 2 — Imports & configuration

In [2]:
import os
import io
from dotenv import load_dotenv
load_dotenv()  # loads .env file if present
import json
import warnings
warnings.filterwarnings('ignore')

# Core ML
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# OpenCV
import cv2

# Hugging Face
from transformers import pipeline, AutoFeatureExtractor, AutoModelForImageClassification

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# NLTK
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# HTTP + FastAPI
import requests
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import uvicorn
import nest_asyncio

# ── CONFIGURATION ──────────────────────────────────────────────
NVIDIA_API_KEY  = os.getenv("NVIDIA_API_KEY", "")   # set via environment variable or .env   
NVIDIA_API_URL  = "https://integrate.api.nvidia.com/v1"
LLAMA_MODEL     = "meta/llama-3.1-70b-instruct" 
HF_TOKEN = os.getenv("HF_TOKEN", "")                # set via environment variable or .env

# Hugging Face model for chest X-ray classification
# Using google/vit-base-patch16-224 as a general vision model
# For production: replace with a chest-X-ray-specific model like
# 'nickmuchi/vit-base-patch16-224-finetuned-chest-xray'
os.environ["HF_TOKEN"] = HF_TOKEN
HF_MODEL_NAME   = "nickmuchi/vit-finetuned-chest-xray-pneumonia"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print("Imports OK ✓")

Device: cpu
Imports OK ✓


## Cell 3 — OpenCV preprocessing module

In [3]:
class ImagePreprocessor:
    """
    Converts a raw medical scan into a model-ready tensor.
    Steps: CLAHE enhancement → denoise → resize → normalize.
    """

    TARGET_SIZE = (224, 224)

    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        self.transform = transforms.Compose([
            transforms.Resize(self.TARGET_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def load_from_bytes(self, image_bytes: bytes) -> np.ndarray:
        """Decode image bytes to BGR numpy array."""
        arr = np.frombuffer(image_bytes, np.uint8)
        return cv2.imdecode(arr, cv2.IMREAD_COLOR)

    def enhance(self, img: np.ndarray) -> np.ndarray:
        """Apply CLAHE on L-channel to boost local contrast."""
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l_eq = self.clahe.apply(l)
        lab_eq = cv2.merge([l_eq, a, b])
        return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

    def denoise(self, img: np.ndarray) -> np.ndarray:
        """Non-local means denoising — good for CT/MRI noise."""
        return cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

    def to_tensor(self, img: np.ndarray) -> torch.Tensor:
        """BGR numpy → RGB PIL → normalised tensor."""
        pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        return self.transform(pil).unsqueeze(0).to(DEVICE)

    def process(self, image_bytes: bytes) -> tuple[torch.Tensor, np.ndarray]:
        """Full pipeline. Returns (tensor, enhanced_bgr_array)."""
        img = self.load_from_bytes(image_bytes)
        img = self.enhance(img)
        img = self.denoise(img)
        return self.to_tensor(img), img


preprocessor = ImagePreprocessor()
print("ImagePreprocessor ready ✓")

ImagePreprocessor ready ✓


## Cell 4 — Hugging Face + PyTorch feature extractor

In [4]:
class MedicalImageAnalyser:
    """
    Two-stage analyser:
      Stage 1 (Hugging Face)  — ImageClassification pipeline for top-k labels
      Stage 2 (PyTorch ResNet) — 512-dim feature vector for anomaly scoring
    """

    def __init__(self, hf_model: str = HF_MODEL_NAME):
        print(f"Loading HuggingFace model: {hf_model} ...")
        self.classifier = pipeline(
            "image-classification",
            model=hf_model,
            device=0 if DEVICE == "cuda" else -1
        )

        # ResNet-18 as lightweight feature backbone (no top layer)
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])
        self.feature_extractor.eval().to(DEVICE)
        print("Analyser ready ✓")

    def classify(self, pil_image: Image.Image, top_k: int = 5) -> list[dict]:
        """Return top-k HuggingFace classification results."""
        return self.classifier(pil_image, top_k=top_k)

    def extract_features(self, tensor: torch.Tensor) -> np.ndarray:
        """Return flattened 512-dim feature vector from ResNet backbone."""
        with torch.no_grad():
            feats = self.feature_extractor(tensor)
        return feats.squeeze().cpu().numpy()

    def analyse(self, tensor: torch.Tensor, bgr_img: np.ndarray) -> dict:
        """Run both stages, return combined findings dict."""
        # Convert tensor back to PIL for HF pipeline
        pil = Image.fromarray(cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB))
        pil = pil.resize((224, 224))

        labels   = self.classify(pil)
        features = self.extract_features(tensor)

        return {
            "top_labels": [{"label": r["label"], "score": round(r["score"], 4)} for r in labels],
            "feature_vector": features,  # passed to anomaly scorer
        }


analyser = MedicalImageAnalyser()
print("MedicalImageAnalyser ready ✓")

Loading HuggingFace model: nickmuchi/vit-finetuned-chest-xray-pneumonia ...


Loading weights: 100%|███████████████████████████████| 200/200 [00:00<00:00, 26947.02it/s]


Analyser ready ✓
MedicalImageAnalyser ready ✓


## Cell 5 — Scikit-learn anomaly scorer

In [5]:
class AnomalyScorer:
    """
    IsolationForest trained on synthetic 'normal' embeddings.
    In production: replace synthetic data with your own labelled scan embeddings.
    """

    def __init__(self, n_normal_samples: int = 200, feature_dim: int = 512):
        np.random.seed(42)
        # Simulate a distribution of 'normal' scan feature vectors
        normal_data = np.random.randn(n_normal_samples, feature_dim) * 0.5

        self.scaler = StandardScaler()
        scaled = self.scaler.fit_transform(normal_data)

        self.model = IsolationForest(contamination=0.05, random_state=42)
        self.model.fit(scaled)
        print("AnomalyScorer ready ✓")

    def score(self, feature_vector: np.ndarray) -> dict:
        """
        Returns anomaly_score (lower = more anomalous) and
        a human-readable risk_level string.
        """
        vec = self.scaler.transform(feature_vector.reshape(1, -1))
        score = self.model.decision_function(vec)[0]   # negative = anomalous
        prediction = self.model.predict(vec)[0]         # -1 = anomaly, 1 = normal

        if prediction == -1:
            risk = "high"    if score < -0.15 else "medium"
        else:
            risk = "low"

        return {
            "anomaly_score": round(float(score), 4),
            "risk_level": risk,
            "is_anomaly": bool(prediction == -1)
        }


anomaly_scorer = AnomalyScorer()
print("AnomalyScorer ready ✓")

AnomalyScorer ready ✓
AnomalyScorer ready ✓


## Cell 6 — NLTK prompt builder

In [6]:
from nltk.tokenize import word_tokenize

class PromptBuilder:
    """
    Takes structured findings and builds a clean LLaMA prompt.
    NLTK is used to:
      - tokenise and deduplicate label words
      - extract key terms for the prompt context
    """

    SYSTEM_PROMPT = (
        "You are an expert radiologist assistant. "
        "Given structured image analysis findings, generate a concise, professional "
        "radiology report in plain English. "
        "Structure your report with: FINDINGS, IMPRESSION, and RECOMMENDATION sections. "
        "Always note this is an AI-assisted report for review by a qualified clinician."
    )

    def build(self, findings: dict, anomaly: dict, scan_type: str = "chest X-ray") -> dict:
        """
        findings  : output from MedicalImageAnalyser.analyse()
        anomaly   : output from AnomalyScorer.score()
        scan_type : free-text description of the scan modality
        """
        # Extract unique meaningful terms via NLTK
        raw_labels = " ".join([r["label"] for r in findings["top_labels"]])
        tokens     = word_tokenize(raw_labels.lower())
        key_terms  = list(dict.fromkeys(tokens))  # deduplicate, preserve order

        # Build structured user message
        label_lines = "\n".join(
            f"  - {r['label']}: confidence {r['score']:.1%}"
            for r in findings["top_labels"]
        )

        user_message = (
            f"Scan type: {scan_type}\n\n"
            f"Image classification results:\n{label_lines}\n\n"
            f"Anomaly analysis:\n"
            f"  - Risk level : {anomaly['risk_level'].upper()}\n"
            f"  - Anomaly detected: {'Yes' if anomaly['is_anomaly'] else 'No'}\n"
            f"  - Anomaly score: {anomaly['anomaly_score']}\n\n"
            f"Key visual terms identified: {', '.join(key_terms[:10])}\n\n"
            "Please generate a structured radiology report based on these findings."
        )

        return {
            "system": self.SYSTEM_PROMPT,
            "user":   user_message
        }


prompt_builder = PromptBuilder()
print("PromptBuilder ready ✓")

PromptBuilder ready ✓


## Cell 7 — LLaMA report generator (NVIDIA API)

In [12]:
class LLaMAReportGenerator:
    """
    Calls the NVIDIA-hosted LLaMA model via their OpenAI-compatible
    /v1/chat/completions endpoint.
    """

    def __init__(self, api_key: str, api_url: str, model: str):
        self.api_key = api_key
        self.api_url = api_url
        self.model   = model
        self.headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type":  "application/json"
        }

    def generate(self, prompt: dict, max_tokens: int = 600) -> str:
        """
        prompt: dict with 'system' and 'user' keys (from PromptBuilder)
        Returns the generated report as a string.
        """
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": prompt["system"]},
                {"role": "user",   "content": prompt["user"]}
            ],
            "max_tokens":   max_tokens,
            "temperature":  0.3,   # low temp for consistent medical reports
            "top_p":        0.9
        }

        try:
            resp = requests.post(
                self.api_url,
                headers=self.headers,
                json=payload,
                timeout=60
            )
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"]
        except requests.exceptions.RequestException as e:
            return f"[API Error] {str(e)}"

# Instantiate all modules 
preprocessor     = ImagePreprocessor()        
analyser         = MedicalImageAnalyser()     
anomaly_scorer   = AnomalyScorer()            
prompt_builder   = PromptBuilder()            
report_generator = LLaMAReportGenerator(
    api_key=NVIDIA_API_KEY,
    api_url=NVIDIA_API_URL + "/chat/completions",   # ← add /chat/completions here
    model=LLAMA_MODEL
)
print("✅ All modules ready!")
print("LLaMAReportGenerator ready ✓")

Loading HuggingFace model: nickmuchi/vit-finetuned-chest-xray-pneumonia ...


Loading weights: 100%|███████████████████████████████| 200/200 [00:00<00:00, 15155.57it/s]


Analyser ready ✓
AnomalyScorer ready ✓
✅ All modules ready!
LLaMAReportGenerator ready ✓


## Cell 8 — Full pipeline (end-to-end test)

In [13]:
def run_pipeline(image_bytes: bytes, scan_type: str = "chest X-ray") -> dict:
    """
    Complete MediScan pipeline.

    Args:
        image_bytes : raw image bytes (JPEG/PNG)
        scan_type   : description string shown in the report

    Returns:
        dict with keys: scan_type, top_labels, anomaly, report
    """
    print("[1/4] Preprocessing image...")
    tensor, bgr_img = preprocessor.process(image_bytes)

    print("[2/4] Analysing image...")
    findings = analyser.analyse(tensor, bgr_img)

    print("[3/4] Scoring anomalies...")
    anomaly = anomaly_scorer.score(findings["feature_vector"])

    print("[4/4] Generating report via LLaMA...")
    prompt = prompt_builder.build(findings, anomaly, scan_type)
    report = report_generator.generate(prompt)

    return {
        "scan_type":  scan_type,
        "top_labels": findings["top_labels"],
        "anomaly":    anomaly,
        "report":     report
    }


# ── Quick smoke test with a synthetic grayscale image ─────────────
# Replace this block with: with open('your_scan.jpg', 'rb') as f: img_bytes = f.read()
noise_img = (np.random.rand(256, 256, 3) * 255).astype(np.uint8)
_, img_bytes = cv2.imencode('.jpg', noise_img)
img_bytes = img_bytes.tobytes()

result = run_pipeline(img_bytes, scan_type="chest X-ray (demo)")

print("\n" + "="*60)
print("PIPELINE OUTPUT")
print("="*60)
print(f"Scan type : {result['scan_type']}")
print(f"Risk level: {result['anomaly']['risk_level'].upper()}")
print(f"Top label : {result['top_labels'][0]['label']} ({result['top_labels'][0]['score']:.1%})")
print("\n--- GENERATED REPORT ---\n")
print(result['report'])

[1/4] Preprocessing image...
[2/4] Analysing image...
[3/4] Scoring anomalies...
[4/4] Generating report via LLaMA...

PIPELINE OUTPUT
Scan type : chest X-ray (demo)
Risk level: MEDIUM
Top label : PNEUMONIA (58.4%)

--- GENERATED REPORT ---

**Chest X-ray Report (Demo)**

**FINDINGS:**
A chest X-ray was performed, and the image analysis results suggest the presence of pneumonia with a confidence level of 58.4%. However, there is also a possibility of a normal chest X-ray with a confidence level of 41.6%. Anomaly analysis indicates a medium risk level with an anomaly score of -0.0074.

**IMPRESSION:**
The findings are suggestive of pneumonia, but the confidence level is not definitive. The presence of pneumonia cannot be ruled out, and further evaluation may be necessary to confirm the diagnosis.

**RECOMMENDATION:**
Given the medium risk level and the suggestive findings of pneumonia, it is recommended that the patient undergo further diagnostic evaluation, such as a follow-up chest X-

## Cell 9 — FastAPI app definition

In [14]:
app = FastAPI(
    title="MediScan AI",
    description="Medical image analysis + radiology report generation",
    version="1.0.0"
)


@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE, "model": LLAMA_MODEL}


@app.post("/analyze")
async def analyze(
    file: UploadFile = File(..., description="Medical scan image (JPEG or PNG)"),
    scan_type: str   = "chest X-ray"
):
    """
    Upload a scan, receive a structured radiology report.

    Returns JSON with:
    - scan_type    : echoed back
    - top_labels   : list of {label, score}
    - anomaly      : {anomaly_score, risk_level, is_anomaly}
    - report       : full LLaMA-generated radiology report text
    """
    image_bytes = await file.read()
    result = run_pipeline(image_bytes, scan_type=scan_type)
    # Remove feature_vector before returning (not serialisable / not useful to client)
    return JSONResponse(content=result)


print("FastAPI app defined ✓")
print("Routes:")
print("  GET  /health")
print("  POST /analyze   (multipart: file, scan_type)")

FastAPI app defined ✓
Routes:
  GET  /health
  POST /analyze   (multipart: file, scan_type)


## Cell 10 — Launch the server

Run this cell to start the API inside the notebook.

Then test it:
```bash
curl -X POST http://localhost:8000/analyze \
  -F 'file=@/path/to/scan.jpg' \
  -F 'scan_type=chest X-ray'
```

In [ ]:
import asyncio
nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)

print("Starting server at http://localhost:8000")
print("Docs available at http://localhost:8000/docs")
print("Press the stop (■) button in Jupyter to shut down.")

await server.serve()

Starting server at http://localhost:8000
Docs available at http://localhost:8000/docs
Press the stop (■) button in Jupyter to shut down.


## Cell 11 — Test client (optional, run in a separate cell while server is live)

In [ ]:
# Run this cell independently (while server is running in Cell 10)

import requests as req

# Health check
health = req.get("http://localhost:8000/health").json()
print("Health:", health)

# Analyze a real scan — change path to your own image
# with open("/path/to/your/scan.jpg", "rb") as f:
#     response = req.post(
#         "http://localhost:8000/analyze",
#         files={"file": ("scan.jpg", f, "image/jpeg")},
#         data={"scan_type": "chest X-ray"}
#     )
#     print(json.dumps(response.json(), indent=2))

---
## Next steps

| Step | What to do |
|------|------------|
| Real HF model | Replace `HF_MODEL_NAME` with a chest-X-ray fine-tuned model (e.g. `nickmuchi/vit-base-patch16-224-finetuned-chest-xray`) |
| Real anomaly data | Train `IsolationForest` on actual normal scan embeddings instead of synthetic data |
| NVIDIA key | Paste your key into `NVIDIA_API_KEY` in Cell 2 |
| Frontend | Point any HTML form or React app at `POST /analyze` |
| Disclaimer | Always include clinical review notice — this tool assists, never replaces, a radiologist |